# Kairos Classification Model
## Objective: Predict corrective maintenance in the next 14 days

This notebook implements a model that, for each vehicle, predicts whether a corrective maintenance ("breakdown") will occur in the next 14 days.

---

## Phase 1: Environment Setup and Raw Data Preparation

### Step 1.1: Environment Configuration
Installation and import of all necessary libraries to ensure our environment has all the tools for each project phase.

In [ ]:
# Library imports
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


print(f"Pandas version: {pd.__version__}")
print(f"XGBoost version: {xgb.__version__}")
print(f"Numpy version: {np.__version__}")

### Step 1.2: Initial Loading and Cleaning
Loading the SERVICE_ORDER_BASE.xlsx file and essential basic cleaning to ensure a consistent database.

In [ ]:
# Load the database
try:
    # First, let's check if the file exists in the notebooks/data directory
    file_path = "data/SERVICE_ORDER_BASE.xlsx"
    df_service = pd.read_excel(file_path)
    print(f" Database loaded successfully with {len(df_service):,} records.")
    print(f" File loaded from: {file_path}")
except FileNotFoundError:
    try:
        # Try in the root directory
        file_path = "../SERVICE_ORDER_BASE.xlsx"
        df_service = pd.read_excel(file_path)
        print(f" Database loaded successfully with {len(df_service):,} records.")
        print(f" File loaded from: {file_path}")
    except FileNotFoundError:
        print(" ERROR: The file 'SERVICE_ORDER_BASE.xlsx' was not found.")
        print(" Check if the file is in:")
        print("   - notebooks/data/SERVICE_ORDER_BASE.xlsx")
        print("   - or in the project root directory")
        raise FileNotFoundError("File SERVICE_ORDER_BASE.xlsx not found")

# Display basic information about the dataset
print(f"\n Dataset Information:")
print(f"   - Rows: {df_service.shape[0]:,}")
print(f"   - Columns: {df_service.shape[1]}")
print(f"\n Available columns:")
for i, col in enumerate(df_service.columns, 1):
    print(f"   {i:2d}. {col}")

In [ ]:
# Initial cleaning
print(" Starting initial data cleaning...")

# Clean column names (remove extra spaces)
df_service.columns = df_service.columns.str.strip()

# Check if essential columns exist
required_columns = ['ASSET CODE', 'SERVICE ORDER ORIGINAL DATE', 'COUNTER  OF SERVICE ORDER', 'PREVENTIVE_CORRECTIVE MAINTENANCE']
missing_columns = [col for col in required_columns if col not in df_service.columns]

if missing_columns:
    print(f" ERROR: Essential columns not found: {missing_columns}")
    print("\n Available columns in dataset:")
    for col in df_service.columns:
        print(f"   - {col}")
    raise ValueError("Essential columns not found in dataset")

print(" All essential columns found!")

# Data type conversions
print("\n Converting data types...")
# CORRECTION: Convert date from YYYYMMDD format (e.g., 20210618) to datetime
print(" Fixing date conversion...")
print(f"   - Original date example: {df_service['SERVICE ORDER ORIGINAL DATE'].iloc[0]}")

# Convert date from YYYYMMDD format to datetime
df_service['SERVICE ORDER ORIGINAL DATE'] = pd.to_datetime(
    df_service['SERVICE ORDER ORIGINAL DATE'].astype(str), 
    format='%Y%m%d',
    errors='coerce'
)

# Check if conversion worked
null_dates = df_service['SERVICE ORDER ORIGINAL DATE'].isnull().sum()
if null_dates > 0:
    print(f"      {null_dates} dates could not be converted")
else:
    print("     All dates converted successfully!")

# Check the result
print(f"   - Converted date example: {df_service['SERVICE ORDER ORIGINAL DATE'].iloc[0]}")
print(f"   - Column type: {df_service['SERVICE ORDER ORIGINAL DATE'].dtype}")

# Convert counter (odometer)
df_service['COUNTER  OF SERVICE ORDER'] = pd.to_numeric(
    df_service['COUNTER  OF SERVICE ORDER'], 
    errors='coerce'
)

# Convert total cost if it exists
if 'GRAND TOTAL' in df_service.columns:
    df_service['GRAND TOTAL'] = pd.to_numeric(
        df_service['GRAND TOTAL'], 
        errors='coerce'
    )

print(" Type conversions completed!")

In [ ]:
# Initial data analysis before cleaning
print(" Initial data analysis BEFORE cleaning:")
print(f"   - Total records: {len(df_service):,}")
print(f"   - Records with null ASSET CODE: {df_service['ASSET CODE'].isnull().sum():,}")
print(f"   - Records with null date: {df_service['SERVICE ORDER ORIGINAL DATE'].isnull().sum():,}")
print(f"   - Records with null counter: {df_service['COUNTER  OF SERVICE ORDER'].isnull().sum():,}")

# Remove records without minimum information for analysis
print("\n Removing records with missing essential information...")

df_service_clean = df_service.dropna(subset=[
    'ASSET CODE', 
    'SERVICE ORDER ORIGINAL DATE', 
    'COUNTER  OF SERVICE ORDER'
]).copy()

print("\n Analysis AFTER cleaning:")
print(f"   - Records kept: {len(df_service_clean):,}")
print(f"   - Records removed: {len(df_service) - len(df_service_clean):,}")
print(f"   - Percentage kept: {len(df_service_clean)/len(df_service)*100:.1f}%")

# Data period analysis
min_date = df_service_clean['SERVICE ORDER ORIGINAL DATE'].min()
max_date = df_service_clean['SERVICE ORDER ORIGINAL DATE'].max()
print(f"\n Data period:")
print(f"   - Earliest date: {min_date.strftime('%m/%d/%Y')}")
print(f"   - Latest date: {max_date.strftime('%m/%d/%Y')}")
print(f"   - Total period: {(max_date - min_date).days} days")

# Maintenance type analysis
print(f"\n Maintenance type distribution:")
maintenance_dist = df_service_clean['PREVENTIVE_CORRECTIVE MAINTENANCE'].value_counts()
for mtype, count in maintenance_dist.items():
    pct = count / len(df_service_clean) * 100
    print(f"   - {mtype}: {count:,} ({pct:.1f}%)")

# Unique vehicle analysis
n_vehicles = df_service_clean['ASSET CODE'].nunique()
print(f"\n Vehicle analysis:")
print(f"   - Unique vehicles: {n_vehicles:,}")
print(f"   - Average records per vehicle: {len(df_service_clean)/n_vehicles:.1f}")

# Update main dataframe
df_service = df_service_clean.copy()

print("\n Initial cleaning completed successfully!")
print(f" Final dataset: {len(df_service):,} records from {n_vehicles:,} vehicles")

# Phase 2: Feature Engineering and Target Creation

This is the most critical phase, where we create the information that the model will use to make predictions.

## Step 2.1: Target Variable Creation (Will_Break_14_Days)

**Objective**: Label each maintenance event with 1 (if a breakdown occurred in the next 14 days) or 0 (if it didn't occur).

**Why**: To teach the model, we need to give it the "correct answer" for historical data.

In [ ]:
print(" Starting target variable creation...")

# Sorting data is crucial for the "look into the future" logic
print("   - Sorting data by vehicle and date...")
df_service.sort_values(by=['ASSET CODE', 'SERVICE ORDER ORIGINAL DATE'], inplace=True)
df_service.reset_index(drop=True, inplace=True)

# Identify dates of all breakdowns (corrective maintenance)
print("   - Identifying breakdown dates...")
df_breakdowns = df_service[
    df_service['PREVENTIVE_CORRECTIVE MAINTENANCE'] == 'CORRECTIVE'
][['ASSET CODE', 'SERVICE ORDER ORIGINAL DATE']].drop_duplicates()

print(f"   - Total unique breakdowns found: {len(df_breakdowns):,}")

# Create a dictionary for fast lookup of breakdown dates by vehicle
print("   - Creating breakdown mapping by vehicle...")
breakdowns_dict = df_breakdowns.groupby('ASSET CODE')['SERVICE ORDER ORIGINAL DATE'].apply(list).to_dict()

print(f"   - Vehicles with breakdown history: {len(breakdowns_dict):,}")
print(f"   - Average breakdowns per vehicle: {len(df_breakdowns)/len(breakdowns_dict):.1f}")

In [ ]:
# Function to apply labeling logic
def check_future_breakdown(row, breakdowns_map, window_days=14):
    """
    Checks if there will be a breakdown in the next N days after the current date.
    
    Args:
        row: DataFrame row
        breakdowns_map: Dictionary with breakdown dates by vehicle
        window_days: Number of days to look into the future (default: 14)
    
    Returns:
        1 if there will be a breakdown in the window, 0 otherwise
    """
    asset_code = row['ASSET CODE']
    current_date = row['SERVICE ORDER ORIGINAL DATE']
    
    # If the vehicle has no breakdown history, it won't break in the future (in this database)
    if asset_code not in breakdowns_map:
        return 0
    
    # Look for a breakdown in the 14-day window
    limit_date = current_date + pd.Timedelta(days=window_days)
    
    for breakdown_date in breakdowns_map[asset_code]:
        # The breakdown must be AFTER the current date and WITHIN the window
        if current_date < breakdown_date <= limit_date:
            return 1  # Found a breakdown in the window
    
    return 0  # No breakdown found in the window

print(" Applying labeling logic...")
print("   - This operation may take a few minutes for large datasets...")

# Apply the function to create the target column
df_service['Will_Break_14_Days'] = df_service.apply(
    check_future_breakdown, 
    axis=1, 
    args=(breakdowns_dict,)
)

print(" Target variable creation completed!")

In [ ]:
# Target variable distribution analysis
print(" Target Variable Analysis:")
print("\n Distribution of 'Will_Break_14_Days' variable:")

distribution = df_service['Will_Break_14_Days'].value_counts().sort_index()
distribution_pct = df_service['Will_Break_14_Days'].value_counts(normalize=True).sort_index() * 100

for value in [0, 1]:
    count = distribution.get(value, 0)
    pct = distribution_pct.get(value, 0)
    label = "WILL NOT break" if value == 0 else "WILL break"
    print(f"   - {value} ({label}): {count:,} records ({pct:.1f}%)")

# Calculate imbalance ratio
breakdowns = distribution.get(1, 0)
no_breakdowns = distribution.get(0, 0)
if breakdowns > 0:
    ratio = no_breakdowns / breakdowns
    print(f"\n Imbalance ratio: {ratio:.1f}:1 (no-breakdown:breakdown)")
    
    if ratio > 10:
        print("      Dataset very imbalanced - will need treatment in the model")
    elif ratio > 5:
        print("      Dataset moderately imbalanced")
    else:
        print("     Dataset relatively balanced")

# Analysis by maintenance type
print("\n Distribution by maintenance type:")
crosstab = pd.crosstab(
    df_service['PREVENTIVE_CORRECTIVE MAINTENANCE'], 
    df_service['Will_Break_14_Days'], 
    margins=True
)
print(crosstab)

print("\n Interpretation:")
print("   - Class 0: Records where NO breakdown occurred in the next 14 days")
print("   - Class 1: Records where a breakdown OCCURRED in the next 14 days")
print("   - The model will learn to identify patterns that precede breakdowns")

## Step 2.2: Predictive Features Creation

**Objective**: Create the "clues" that the model will use. For each row, features should describe the vehicle's state and history up to that moment.

**Why**: Feature quality determines model performance.

In [ ]:
print(" Starting predictive feature engineering...")
# Usage and Wear Features
print("\n Creating usage and wear features...")

# Current odometer (rename for clarity)
df_service['Current_Odometer'] = df_service['COUNTER  OF SERVICE ORDER']

# Mileage since last maintenance
df_service['KM_Since_Last_Maintenance'] = df_service.groupby('ASSET CODE')['COUNTER  OF SERVICE ORDER'].diff().fillna(0)

# Ensure we don't have negative values (can happen if odometer was reset)
df_service['KM_Since_Last_Maintenance'] = df_service['KM_Since_Last_Maintenance'].clip(lower=0)

print("     Usage features created")

# Maintenance History Features
print("\n Creating maintenance history features...")

# Total number of services up to this point (cumulative)
df_service['Total_Services'] = df_service.groupby('ASSET CODE').cumcount() + 1

# Total number of breakdowns up to this point (cumulative)
df_service['Total_Breakdowns'] = df_service.groupby('ASSET CODE')['PREVENTIVE_CORRECTIVE MAINTENANCE'].transform(
    lambda x: (x == 'CORRECTIVE').cumsum()
)

# Last maintenance cost (shift to not leak future information)
if 'GRAND TOTAL' in df_service.columns:
    df_service['Last_Maintenance_Cost'] = df_service.groupby('ASSET CODE')['GRAND TOTAL'].shift(1).fillna(0)
else:
    df_service['Last_Maintenance_Cost'] = 0
    print("      GRAND TOTAL column not found - using 0 for costs")
# Days since last maintenance
df_service['Days_Since_Last_Maintenance'] = df_service.groupby('ASSET CODE')['SERVICE ORDER ORIGINAL DATE'].diff().dt.days.fillna(0)

print("     History features created")

In [ ]:
# Breakdown-specific features
print("\n Creating breakdown-specific features...")

# Isolate breakdowns to calculate specific features
df_breakdown_hist = df_service[df_service['PREVENTIVE_CORRECTIVE MAINTENANCE'] == 'CORRECTIVE'].copy()

if len(df_breakdown_hist) > 0:
    # Calculate days between consecutive breakdowns
    df_breakdown_hist['Days_Between_Breakdowns'] = df_breakdown_hist.groupby('ASSET CODE')['SERVICE ORDER ORIGINAL DATE'].diff().dt.days
    
    # For each vehicle, get the last value of "days since last breakdown"
    map_days_since_breakdown = df_breakdown_hist.groupby('ASSET CODE')['Days_Between_Breakdowns'].last()
    
    # Map to main database
    df_service['Days_Since_Last_Breakdown'] = df_service['ASSET CODE'].map(map_days_since_breakdown).fillna(999)  # 999 = never broke
    
    print(f"     Breakdown features created for {len(map_days_since_breakdown)} vehicles")
else:
    df_service['Days_Since_Last_Breakdown'] = 999
    print("      No breakdowns found - using default value")

# Additional useful features
print("\n Creating additional features...")

# Breakdown rate (breakdowns / total services)
df_service['Breakdown_Rate'] = df_service['Total_Breakdowns'] / df_service['Total_Services']
df_service['Breakdown_Rate'] = df_service['Breakdown_Rate'].fillna(0)

# Record age (days since vehicle's first record)
df_service['First_Date'] = df_service.groupby('ASSET CODE')['SERVICE ORDER ORIGINAL DATE'].transform('min')
df_service['Vehicle_Age_Days'] = (df_service['SERVICE ORDER ORIGINAL DATE'] - df_service['First_Date']).dt.days

# Usage intensity (services per day)
df_service['Usage_Intensity'] = df_service['Total_Services'] / (df_service['Vehicle_Age_Days'] + 1)  # +1 to avoid division by zero

print("     Additional features created")

print("\n Feature engineering completed!")

In [ ]:
# Analysis of created features
print(" Analysis of Created Features:")

# List of created features
feature_columns = [
    'Current_Odometer', 'KM_Since_Last_Maintenance', 'Total_Services',
    'Total_Breakdowns', 'Last_Maintenance_Cost', 'Days_Since_Last_Maintenance',
    'Days_Since_Last_Breakdown', 'Breakdown_Rate', 'Vehicle_Age_Days', 'Usage_Intensity'
]

print(f"\n Total features created: {len(feature_columns)}")

# Feature statistics
print("\n Feature Statistics:")
feature_stats = df_service[feature_columns].describe()
print(feature_stats.round(2))

# Check for missing values in features
print("\n Missing values in features:")
missing_values = df_service[feature_columns].isnull().sum()
if missing_values.sum() > 0:
    for feature, missing in missing_values.items():
        if missing > 0:
            pct = missing / len(df_service) * 100
            print(f"   - {feature}: {missing:,} ({pct:.1f}%)")
else:
    print("    No missing values found in features")

# Feature correlation with target
print("\n Feature correlation with target:")
correlations = df_service[feature_columns + ['Will_Break_14_Days']].corr()['Will_Break_14_Days'].drop('Will_Break_14_Days')
correlations_sorted = correlations.abs().sort_values(ascending=False)

print("   Top features by correlation with target:")
for feature, corr in correlations_sorted.head(5).items():
    print(f"   - {feature}: {corr:.3f}")

print("\n Feature analysis completed!")
print(f" Dataset ready for modeling: {len(df_service):,} records with {len(feature_columns)} features")

# Phase 3: Modeling, Validation and Evaluation

This phase implements XGBoost model training, temporal validation and detailed evaluation with appropriate metrics for the classification problem.

## Step 3.1: Final Selection and Data Split

**Objective**: Select final columns for the model and split them into train and test using temporal approach.

**Why**: Ensures the model is trained on the past and tested on the future, simulating reality.

In [ ]:
print(" Starting final selection and data split...")

# Select final columns for the model
features = [
    'Current_Odometer',
    'KM_Since_Last_Maintenance',
    'Total_Services',
    'Total_Breakdowns',
    'Last_Maintenance_Cost',
    'Days_Since_Last_Maintenance',
    'Days_Since_Last_Breakdown',
    'Breakdown_Rate',
    'Vehicle_Age_Days',
    'Usage_Intensity'
]

target = 'Will_Break_14_Days'

print(f" Selected features: {len(features)}")
for i, feature in enumerate(features, 1):
    print(f"   {i:2d}. {feature}")

# Create final dataset removing records with null values
df_final_model = df_service[features + [target, 'SERVICE ORDER ORIGINAL DATE', 'ASSET CODE']].dropna()

print(f"\n Dataset for modeling:")
print(f"   - Total records: {len(df_service):,}")
print(f"   - Records after cleaning: {len(df_final_model):,}")
print(f"   - Records removed: {len(df_service) - len(df_final_model):,}")
print(f"   - Percentage kept: {len(df_final_model)/len(df_service)*100:.1f}%")

In [ ]:

# Temporal data split
print("\n Performing temporal data split...")

# Use 80% of oldest data for training and 20% most recent for testing
cutoff_date = df_final_model['SERVICE ORDER ORIGINAL DATE'].quantile(0.8)

df_train = df_final_model[df_final_model['SERVICE ORDER ORIGINAL DATE'] < cutoff_date]
df_test = df_final_model[df_final_model['SERVICE ORDER ORIGINAL DATE'] >= cutoff_date]

# Separate features and target
X_train = df_train[features]
y_train = df_train[target]
X_test = df_test[features]
y_test = df_test[target]

print(f" Temporal split performed on {cutoff_date.strftime('%m/%d/%Y')}:")
print(f"   - Train: {len(X_train):,} records ({len(X_train)/len(df_final_model)*100:.1f}%)")
print(f"   - Test:  {len(X_test):,} records ({len(X_test)/len(df_final_model)*100:.1f}%)")

# Analysis of target variable distribution in each set
print(f"\n Target variable distribution:")
print("TRAIN:")
train_dist = y_train.value_counts().sort_index()
for value in [0, 1]:
    count = train_dist.get(value, 0)
    pct = count / len(y_train) * 100
    label = "No breakdown" if value == 0 else "Breakdown"
    print(f"   - {label}: {count:,} ({pct:.1f}%)")

print("TEST:")
test_dist = y_test.value_counts().sort_index()
for value in [0, 1]:
    count = test_dist.get(value, 0)
    pct = count / len(y_test) * 100
    label = "No breakdown" if value == 0 else "Breakdown"
    print(f"   - {label}: {count:,} ({pct:.1f}%)")

# Check if there are unique vehicles in train and test
vehicles_train = set(df_train['ASSET CODE'].unique())
vehicles_test = set(df_test['ASSET CODE'].unique())
vehicles_common = vehicles_train.intersection(vehicles_test)

print(f"\n Vehicle analysis:")
print(f"   - Unique vehicles in train: {len(vehicles_train):,}")
print(f"   - Unique vehicles in test: {len(vehicles_test):,}")
print(f"   - Vehicles in both sets: {len(vehicles_common):,}")
print(f"   - Overlap: {len(vehicles_common)/len(vehicles_train.union(vehicles_test))*100:.1f}%")

## Step 3.2: XGBoost Training

**Objective**: Train the XGBClassifier model, handling data imbalance.

**Why**: Imbalance treatment is crucial for the model to learn to identify the rare class (breakdowns).

In [ ]:
print(" Starting XGBoost model training...")

# Calculate weight for positive class (breakdowns)
# scale_pos_weight = (negative count) / (positive count)
count_negative = (y_train == 0).sum()
count_positive = (y_train == 1).sum()

if count_positive > 0:
    scale_pos_weight = count_negative / count_positive
    print(f" Imbalance treatment:")
    print(f"   - Class 0 (no breakdown): {count_negative:,}")
    print(f"   - Class 1 (breakdown): {count_positive:,}")
    print(f"   - Weight for positive class: {scale_pos_weight:.2f}")
else:
    scale_pos_weight = 1
    print(" No breakdowns found in training set!")

# Instantiate XGBoost with optimized parameters
xgb_classifier = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,  # CRITICAL parameter for imbalance
    use_label_encoder=False,
    eval_metric='logloss',
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1  # Use all available cores
)

print(f"\n Model parameters:")
print(f"   - Estimators: {xgb_classifier.n_estimators}")
print(f"   - Max depth: {xgb_classifier.max_depth}")
print(f"   - Learning rate: {xgb_classifier.learning_rate}")
print(f"   - Positive class weight: {scale_pos_weight:.2f}")

In [ ]:
# Train the model
print("\n Training the model...")
print("   - This operation may take a few minutes...")

import time
start_time = time.time()

# Train the model
xgb_classifier.fit(X_train, y_train)

end_time = time.time()
training_time = end_time - start_time

print(f" Model trained successfully!")
print(f" Training time: {training_time:.1f} seconds")

# Feature importance analysis
print(f"\n Feature Importance (top 5):")
feature_importance = xgb_classifier.feature_importances_
feature_names = X_train.columns

# Create DataFrame for easier analysis
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

for i, (_, row) in enumerate(importance_df.head().iterrows()):
    print(f"   {i+1}. {row['feature']}: {row['importance']:.3f}")

print(f"\n Model ready to make predictions!")

## Step 3.3: Detailed Evaluation

**Objective**: Evaluate the model with correct metrics (Recall and Precision).

**Why**: Accuracy is misleading. We need to know the model's real ability to find breakdowns (Recall) and its false alarm rate (Precision).

In [ ]:
print(" Starting detailed model evaluation...")

# Make predictions on test set
print("   - Making predictions...")
y_pred = xgb_classifier.predict(X_test)
y_pred_proba = xgb_classifier.predict_proba(X_test)[:, 1]  # Probabilities for class 1

print(" Predictions completed!")

print("\n" + "="*60)
print(" MODEL EVALUATION ON TEST SET")
print("="*60)

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)

print(f"\n Confusion Matrix:")
print("                 Predicted")
print("               0       1")
print(f"Actual  0   {cm[0,0]:6d}  {cm[0,1]:6d}")
print(f"        1   {cm[1,0]:6d}  {cm[1,1]:6d}")

# Calculate metrics manually for better explanation
tn, fp, fn, tp = cm.ravel()

print(f"\n Matrix Interpretation:")
print(f"   - True Negatives (TN): {tn:,} - Correctly predicted would NOT break")
print(f"   - False Positives (FP): {fp:,} - Incorrectly predicted would break")
print(f"   - False Negatives (FN): {fn:,} - Incorrectly predicted would NOT break")
print(f"   - True Positives (TP): {tp:,} - Correctly predicted would break")

In [ ]:
# Detailed metrics
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, accuracy_score

print(f"\n DETAILED METRICS:")
print("-" * 40)

# General metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_roc = roc_auc_score(y_test, y_pred_proba)

print(f" Overall Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")
print(f" Precision (Breakdowns): {precision:.3f} ({precision*100:.1f}%)")
print(f" Recall (Breakdowns): {recall:.3f} ({recall*100:.1f}%)")
print(f" F1-Score: {f1:.3f}")
print(f" AUC-ROC: {auc_roc:.3f}")



# Complete report
print(f"\n Complete Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Breakdown', 'Breakdown']))